In [1]:
#python3 -m pip install tqdm pandas spacy
#python -m spacy download fr_core_news_lg

from tqdm import tqdm
import pandas as pd
import spacy
import re

import sys
print(sys.executable)

pd.set_option('display.max_columns', 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

presse = pd.read_csv("/Users/pol/Downloads/press_corpus.csv",
                     encoding="utf-8-sig",
                     sep=";")

nlp = spacy.load('fr_core_news_lg')

/Users/pol/Documents/GitHub/music_artists_data_paper/.venv/bin/python


100%|██████████| 25673/25673 [09:15<00:00, 46.25it/s] 


In [ ]:
df = presse

# create spacy doc object with article_text, keep article id
# parallelize
docs = nlp.pipe(
    zip(df["article_text"], df["article_id"]),
    as_tuples=True,
    disable=['tok2vec', 'parser', 'lemmatizer'],
    batch_size=8,
    n_process=4
)

result = []

# for all ents that are not LOC, extract ents contained in each article
with tqdm(total=len(df)) as pbar:
    for doc, idx in docs: 
        for ent in doc.ents:
            if  ent.label_ != 'LOC':
                result.append((idx, ent.text, ent.label_))
            
        pbar.update(1)
        
ner_df = pd.DataFrame(result, columns=['article_id', 'ent_name', 'ner_type'])
ner_df.to_csv("/Users/pol/Downloads/ner_df_raw.csv", 
                      sep=";",
                      encoding="utf-8-sig")

In [2]:
ner_df = pd.read_csv("/Users/pol/Downloads/ner_df_raw.csv",
                     sep=";",
                      encoding="utf-8-sig")

ner_df = ner_df.drop(columns="Unnamed: 0")
ner_df = ner_df.dropna(subset="ent_name")


# solve tiny issue with names starting with - (that lead to csv encoding error)
def remove_tiretdusix(string):
    string = re.sub(pattern="^-", repl="", string=string)
    string = string.strip()
    return string

ner_df["ent_name"] = ner_df["ent_name"].apply(remove_tiretdusix)

# count occurrences of each name
ner_df["name_count"] = ner_df["ent_name"].map(ner_df["ent_name"].value_counts()) # in total
# ner_df["article_count"] = ner_df.groupby("name")["article_id"].transform("nunique") # in N articles
ner_df


,article_id,ent_name,ner_type,name_count
0,1,Serge Reggiani - Succès & confidences,PER,1
1,1,Reggiani,ORG,32
2,1,Brel,PER,290
3,1,Barbara,PER,551
4,1,Ventura,PER,9
...,...,...,...,...
749728,52481,Moussorgsky,PER,4
749729,52481,Galina Vichnevskaïa,PER,10
749730,52481,Arturo Benedetti Michelangeli,PER,9
749731,52481,Ballade no 1,MISC,2


In [ ]:
# join source again to prepare decomposition into media outlets
ner_df = ner_df.merge(
    presse[["article_id", "source"]],
    on="article_id",
    how="left"
)

# decompose name and article count into media outlet
source_stats = (
    ner_df.groupby(["ent_name", "source"])
    .agg(
        name_count=("ent_name", "size"),
        #article_count=("article_id", "nunique")
    )
)

wide = source_stats.unstack(fill_value=0)
wide.columns = [f"{metric}_{source}" for metric, source in wide.columns]
wide = wide.reset_index()

# df of unique extracted entities
extracted_ents = (
    ner_df
    .groupby("ent_name")
    .agg({"article_id": "first",
          "name_count": "first",
          #"article_count": "first"
          })
    .sort_values(by='name_count', ascending=False)
    .reset_index()
    .merge(wide,
           on="ent_name",
           how="left")
)

extracted_ents["name_id"] = extracted_ents.index + 1

# save
extracted_ents.to_csv("/Users/pol/Downloads/extracted_ents_2105.csv", 
                      sep=";",
                      encoding="utf-8-sig")

extracted_entse

In [ ]:
# press_corpus with column listing all entities for each text
article_text = presse[["article_id", "article_text"]]

press_corpus_ents = (
    ner_df
    .groupby("article_id").agg({
            "ent_name": list,
            #"name_count": "first",
            #"article_count": "first"
        })
    .join(article_text, on="article_id")
)

press_corpus_ents = press_corpus_ents[["article_id", "article_text", "ent_name"]]

press_corpus_ents.to_csv("/Users/pol/Downloads/press_corpus_ents_2105.csv", 
                   sep=";",
                   encoding="utf-8-sig")